In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# USER CONFIG
# =========================
INPUT_FILES = [
    "/mnt/data/gs_combine_final_clean.csv",
    "/mnt/data/jpm_combine_final_clean.csv",
    "/mnt/data/blk_combine_final_clean.csv",
    "/mnt/data/stt_combine_final_clean.csv",
    "/mnt/data/wfc_combine_final_clean.csv",
]

# Shares outstanding constants (copied from your current *_FILLED_liquidity_extended.csv files)
SHARES_OUTSTANDING = {
    "GS":  299_930_000.0,
    "JPM": 2_720_000_000.0,
    "BLK": 155_150_000.0,
    "STT": 279_310_000.0,
    "WFC": 3_140_000_000.0,
}

SHARES_META = {
    "shares_outstanding_source": "Yahoo Finance Key Statistics",
    "shares_outstanding_asof": "2026-01-07",
    "shares_outstanding_confidence": "low",
    "external_identity_status": "ticker_match_ok (SEC validation recommended)",
}

CHANNEL_ORDER = ["Casual Investors", "Mixed Investors", "Professional Investors"]

# =========================
# HELPERS
# =========================
def _compute_rolls_measure_const(mkt: pd.DataFrame) -> dict:
    """Roll (1984): 2*sqrt(-cov(Δp_t, Δp_{t-1})) if cov<0 else NaN, using adjusted prices."""
    out = {}
    for sym, g in mkt.groupby("symbol", sort=False):
        g = g.sort_values("date_dt")
        dp = g["adjusted"].diff()
        cov = dp.cov(dp.shift(1))  # pairwise dropna
        if pd.isna(cov) or cov >= 0:
            out[sym] = np.nan
        else:
            out[sym] = float(2 * np.sqrt(-cov))
    return out

def extend_liquidity_panel(df: pd.DataFrame) -> pd.DataFrame:
    # 1) Drop fully blank rows
    df = df.dropna(subset=["date", "symbol", "channel_classification"], how="any").copy()

    # 2) Standardize date; drop unparseable
    dt = pd.to_datetime(df["date"], errors="coerce")
    df = df.loc[dt.notna()].copy()
    df["date"] = dt[dt.notna()].dt.strftime("%Y-%m-%d")

    # 3) Numeric market columns
    for c in ["open", "high", "low", "close", "volume", "adjusted"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # 4) Unique market table by (symbol, date)
    mcols = ["symbol", "date", "open", "high", "low", "close", "volume", "adjusted"]
    mkt = df[mcols].drop_duplicates(["symbol", "date"]).copy()
    mkt["date_dt"] = pd.to_datetime(mkt["date"])
    mkt = mkt.sort_values(["symbol", "date_dt"])

    # 5) Market log return for proxies
    mkt["r_mkt"] = np.log(mkt["adjusted"]).groupby(mkt["symbol"]).diff()

    # 6) New columns
    denom = (mkt["high"] + mkt["low"]) / 2.0
    mkt["hl_spread"] = (mkt["high"] - mkt["low"]) / denom

    mkt["typical_price"] = (mkt["high"] + mkt["low"] + mkt["close"]) / 3.0
    mkt["vwap_proxy"] = mkt["typical_price"]

    mkt["_pv"] = mkt["vwap_proxy"] * mkt["volume"]
    mkt["vwap_cum"] = mkt.groupby("symbol")["_pv"].cumsum() / mkt.groupby("symbol")["volume"].cumsum()
    mkt.drop(columns=["_pv"], inplace=True)

    mkt["dollar_volume"] = mkt["close"] * mkt["volume"]

    mkt["volume_zero_flag"] = (mkt["volume"] == 0).astype(int)
    mkt["zero_return_flag"] = ((mkt["r_mkt"] == 0) & (mkt["r_mkt"].notna())).astype(int)

    mkt["amihud_impact_proxy"] = np.where(
        (mkt["dollar_volume"] > 0) & (mkt["r_mkt"].notna()),
        np.abs(mkt["r_mkt"]) / mkt["dollar_volume"],
        np.nan,
    )
    mkt["kyle_lambda_proxy"] = np.where(
        (mkt["volume"] > 0) & (mkt["r_mkt"].notna()),
        np.abs(mkt["r_mkt"]) / mkt["volume"],
        np.nan,
    )

    mkt["shares_outstanding"] = mkt["symbol"].map(SHARES_OUTSTANDING)
    mkt["turnover_ratio"] = np.where(
        (mkt["shares_outstanding"] > 0),
        mkt["volume"] / mkt["shares_outstanding"],
        np.nan,
    )

    rm_map = _compute_rolls_measure_const(mkt)
    mkt["rolls_measure"] = mkt["symbol"].map(rm_map)

    mkt["shares_outstanding_source"] = SHARES_META["shares_outstanding_source"]
    mkt["shares_outstanding_asof"] = SHARES_META["shares_outstanding_asof"]
    mkt["shares_outstanding_confidence"] = SHARES_META["shares_outstanding_confidence"]
    mkt["external_identity_status"] = SHARES_META["external_identity_status"]

    mkt["rolls_measure_rolling"] = np.nan
    mkt["rolls_measure_strict"] = np.nan
    mkt["rolls_measure_window"] = 252
    mkt["rolls_measure_min_periods"] = 126

    keep = [
        "symbol","date",
        "hl_spread","zero_return_flag","typical_price","vwap_proxy","vwap_cum","rolls_measure",
        "dollar_volume","kyle_lambda_proxy","amihud_impact_proxy","turnover_ratio","shares_outstanding",
        "shares_outstanding_source","shares_outstanding_asof","shares_outstanding_confidence","external_identity_status",
        "rolls_measure_rolling","rolls_measure_strict","rolls_measure_window","rolls_measure_min_periods",
        "volume_zero_flag",
    ]
    mkt = mkt[keep]

    # 7) Merge back to panel
    out = df.merge(mkt, on=["symbol","date"], how="left", validate="m:1")

    # 8) Sort like your current outputs
    out["date_dt"] = pd.to_datetime(out["date"])
    out["channel_classification"] = pd.Categorical(out["channel_classification"], categories=CHANNEL_ORDER, ordered=True)
    out = out.sort_values(["date_dt","channel_classification"]).drop(columns=["date_dt"]).reset_index(drop=True)

    # 9) Column order
    orig_cols = [
        "symbol","date","open","high","low","close","volume","adjusted",
        "channel_classification","v2tone1","amihud_ratio","amihud_zscore","illiq","r","volatility",
        "intra_day_chg","downside_risk","var1pct","lnvol_chg"
    ]
    new_cols = [c for c in keep if c not in ["symbol","date"]]
    return out[orig_cols + new_cols]

# =========================
# RUN
# =========================
for in_path in INPUT_FILES:
    in_path = Path(in_path)
    df = pd.read_csv(in_path)

    out_df = extend_liquidity_panel(df)

    out_path = in_path.with_name(in_path.stem + "_FILLED_liquidity_extended.csv")
    out_df.to_csv(out_path, index=False)

    print(f"Saved: {out_path} | rows={len(out_df):,} cols={out_df.shape[1]}")
